# Persiann PDIR-Now Data

In [1]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr

In [2]:
from pansat.time import TimeRange
from pansat.utils import resample_data_binned

In [3]:
from pansat.utils import make_latlon_area
lon_min = -125
lat_min = 24
lon_max = -65
lat_max = 51
imerg_grid = make_latlon_area(
    lon_min,
    lat_min,
    lon_max,
    lat_max,
    (lon_max - lon_min) / 0.1, 
    (lat_max - lat_min) / 0.1
)
imerg_grid

/home/simon/miniconda3/envs/chimp/lib/python3.10/site-packages/pyproj/crs/crs.py:1293: UserWarning: You will likely lose important projection information when converting to a PROJ string from another format. See: https://proj.org/faq.html#what-is-the-best-format-for-describing-coordinate-reference-systems
  proj = self._crs.to_proj4(version=version)


Area ID: latlongrid
Description: Regular lat/lon grid
Projection ID: latlon
Projection: {'datum': 'WGS84', 'no_defs': 'None', 'proj': 'longlat', 'type': 'crs'}
Number of columns: 600
Number of rows: 270
Area extent: (-125, 24, -65, 51)

In [6]:
from pansat.products.satellite.persiann import pdirnow_hourly
pdir_files = sorted(list(Path("/edata2/simon/satellite_data/persiann/").glob("pdirnow*.bin.gz")))
pdir_files = {
    pdirnow_hourly.get_temporal_coverage(path).start: path for path in pdir_files
}

In [15]:
from datetime import datetime
from pansat.products.satellite import gpm
from pansat.utils import resample_data_binned

def extract_pdir_now_data(year, month, day, hour) -> xr.Dataset:
    """
    Extract PERSIANN PDIR-Now data over CONUS

    Args:
        year: The year
        month: the month
        day: the day
        hout the hour

    Return:
        The PDIR-Now precip rate for the requested hour.
    """
    time = datetime(year, month, day, hour)
    path = pdir_files[time]
    data = pdirnow_hourly.open(path).rename(precipitation="surface_precip")
    data = resample_data_binned(data, imerg_grid)
    return data
    

In [16]:
output_path = Path("/gdata2/simon/gprof_ir/conus/pdir_now")
output_path.mkdir(parents=True, exist_ok=True)

In [ ]:
start_time = np.datetime64("2022-01-01")
end_time = np.datetime64("2022-07-01")

for hour in np.arange(start_time, end_time, np.timedelta64(1, "h")):
    date = hour.astype("datetime64[s]").item()
    print(date)
    output_file = output_path / date.strftime("pdir_now_%Y%m%d%H%M%S.nc")
    if not output_file.exists():
        try:
            pdir_now_data = extract_pdir_now_data(date.year, date.month, date.day, date.hour)
            for var in ["surface_precip"]:
                pdir_now_data[var].encoding = {
                    "zlib": True,
                    "dtype": "float32"
                }
            pdir_now_data.to_netcdf(output_file)
        except Exception as exc:
            raise exc